# Estimating tumor composition from bulk gene expression

Most of what we know about cancer at the molecular level comes from *bulk*
measurements: grind up a tumor, average the gene activity across all of its cells,
and read out one number per gene. The catch is that a tumor is a mixture. Cancer
cells, immune cells, and connective tissue all get blended together, and the
averaging hides which cells contributed what.

In this lab you will use single-cell data, where each cell is measured on its own,
as a reference to estimate the cellular makeup of about a thousand bulk-measured
breast tumors. Then you will use those estimates to settle a question that comes up
constantly in cancer research: when a gene looks different between two groups of
tumors, is the gene actually changing inside the cells, or do the tumors just
contain different proportions of cells?

No prior Python needed. Run each code cell in order with the play button or
Shift+Enter, and read the text in between. A few cells are marked **Exercise** and
ask you to change one line.

## 1. The data

A tumor is not a solid block of cancer cells. Mixed in among them are T cells and
other immune cells, fibroblasts that build connective tissue, and the endothelial
cells that line blood vessels. Together these make up the tumor microenvironment,
and their proportions matter. The amount of immune infiltration, for example, helps
predict whether immunotherapy will work.

There are two ways to measure gene expression, and they trade off against each
other. Single-cell RNA-seq reads the genes of each cell separately, so it tells us
exactly which cell types are present, but it is expensive and exists for relatively
few tumors. Bulk RNA-seq averages over the whole tumor at once. It is cheap and
available for thousands of patients, but the cell-type detail washes out in the
average.

This lab uses one of each:
- a single-cell **atlas** of breast tumors (Xu et al., 2024), already labeled with
  cell types by experts, which we treat as a reference, and
- bulk RNA-seq from about a thousand breast tumors in **TCGA**, each labeled as one
  of two subtypes: IDC (ductal, the common kind) or ILC (lobular, roughly one case
  in ten).

Run the next two cells to install the libraries and load the data.

In [ ]:
# Colab already has most of these; this guarantees compatible versions.
!pip install -q 'seaborn>=0.13' pandas numpy scipy matplotlib
print('libraries installed')

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import nnls
from scipy.stats import mannwhitneyu
%matplotlib inline
sns.set_style('white')
plt.rcParams['figure.dpi'] = 110

RAW = 'https://raw.githubusercontent.com/bts76-pitt/bioinfo-bootcamp-2026-data/main/'
sig    = pd.read_csv(RAW + 'signature_matrix.csv', index_col=0)   # genes x cell types
viz    = pd.read_csv(RAW + 'atlas_umap.csv.gz')                   # single cells (for plots)
counts = pd.read_csv(RAW + 'atlas_lineage_counts.csv')           # cells per type in the atlas
bulk   = pd.read_csv(RAW + 'tcga_brca_bulk.csv', index_col=0)     # genes x patients
clin   = pd.read_csv(RAW + 'tcga_brca_clinical.csv')             # patient -> subtype

print('Atlas single cells shown :', f'{viz.shape[0]:,}')
print('Patient tumors (bulk)    :', f'{bulk.shape[1]:,}')
print('Cell types in reference  :', list(sig.columns))
print('Subtype counts           :', clin.subtype.value_counts().to_dict())

## 2. Looking at the single cells

The atlas holds 236,363 cells from breast tumors, which experts sorted into seven
broad types. (We merged some closely related subtypes, all T and NK cells for
instance, so the later math stays stable.) Here is how many cells of each type were
measured.

In [ ]:
order = counts.sort_values('n_cells', ascending=False)['lineage'].tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=counts, x='n_cells', y='lineage', order=order, color='#4c72b0', ax=ax)
ax.set(xlabel='number of cells in the atlas', ylabel='', title='Cell types measured in the breast-tumor atlas')
sns.despine(); plt.tight_layout(); plt.show()

### Putting 20,000 cells on a map

Each cell carries readings for about 20,000 genes, far too many to plot directly. A
**UMAP** squeezes all of that down to two dimensions, arranging cells so that ones
with similar gene activity end up near each other. We cannot draw all 236,363 cells,
so the map below shows a sample of about 20,000, each colored by its assigned type.
Cells of the same type land in the same neighborhood, which is the first sign the
labels mean something.

In [ ]:
cell_types = sorted(viz.cell_type.unique())
palette = dict(zip(cell_types, sns.color_palette('tab10', len(cell_types))))
fig, ax = plt.subplots(figsize=(8, 6.5))
for ct in cell_types:
    d = viz[viz.cell_type == ct]
    ax.scatter(d.UMAP1, d.UMAP2, s=4, color=palette[ct], label=ct, alpha=0.6)
ax.legend(markerscale=4, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
ax.set(title='UMAP of single cells, colored by cell type', xticks=[], yticks=[])
sns.despine(left=True, bottom=True); plt.tight_layout(); plt.show()

### What tells the types apart: marker genes

Each cell type runs a characteristic set of genes, its markers. EPCAM is switched on
in epithelial (cancer) cells, CD3D in T cells, COL1A1 in fibroblasts. Color the same
map by a single gene and you can watch it pick out one neighborhood and ignore the
rest. This is essentially how the cell-type labels were assigned in the first place.

In [ ]:
markers = {'EPCAM': 'Cancer Epithelial', 'CD3D': 'T/NK Cells', 'COL1A1': 'Fibroblasts',
           'CD68': 'Myeloid', 'PECAM1': 'Endothelial', 'MS4A1': 'B/Plasma'}
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for ax, (gene, ct) in zip(axes.ravel(), markers.items()):
    s = ax.scatter(viz.UMAP1, viz.UMAP2, s=3, c=viz[gene], cmap='viridis')
    ax.set(title=f'{gene}  (marks {ct})', xticks=[], yticks=[])
    plt.colorbar(s, ax=ax, shrink=0.7)
sns.despine(left=True, bottom=True); plt.tight_layout(); plt.show()

A dot plot says the same thing more compactly. Rows are cell types, columns are
markers. A dot's color is the average expression of that gene in that type, and its
size is the fraction of cells expressing it at all. A clean marker shows up as one
big dark dot in its own row and faded specks everywhere else.

In [ ]:
marker_genes = ['EPCAM', 'KRT8', 'CD3D', 'CD8A', 'COL1A1', 'PDGFRB',
                'CD68', 'LYZ', 'PECAM1', 'VWF', 'MS4A1', 'CD79A', 'RGS5']
rows = []
for ct in cell_types:
    sub = viz[viz.cell_type == ct]
    for g in marker_genes:
        rows.append({'cell_type': ct, 'gene': g,
                     'mean_expr': sub[g].mean(), 'pct': (sub[g] > 0).mean() * 100})
dp = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(dp.gene, dp.cell_type, s=dp.pct * 3.5, c=dp.mean_expr,
                cmap='Reds', edgecolor='0.6', linewidth=0.4)
ax.set(title='Marker genes (color = mean expression, size = % of cells expressing)')
plt.xticks(rotation=45, ha='right')
plt.colorbar(sc, label='mean expression', shrink=0.8)
plt.tight_layout(); plt.show()

**Exercise 1.** Swap in a different gene below and run the cell. Try `'CD8A'` (T
cells), `'VWF'` (endothelial), or `'CDH1'`, a cancer gene we will come back to.
Where does it light up?

In [ ]:
gene = 'EPCAM'        # <-- change this gene, then run the cell
plt.figure(figsize=(6, 5))
s = plt.scatter(viz.UMAP1, viz.UMAP2, s=4, c=viz[gene], cmap='viridis')
plt.colorbar(s, label=f'{gene} expression'); plt.title(gene)
plt.xticks([]); plt.yticks([]); sns.despine(left=True, bottom=True)
plt.tight_layout(); plt.show()

## 3. Turning cells into a reference

To read composition out of bulk data, we first boil each cell type down to a typical
expression profile by averaging all of its cells. Stack those profiles side by side
and you get a **signature matrix**: one row per gene, one column per cell type.

A natural instinct is to use every gene, but that backfires. A handful of very
highly expressed genes end up dominating the later fit, and the estimates collapse
onto a single cell type. So we keep only marker genes, the ones that are distinctly
high in one type. The prepared signature uses the top 50 markers per cell type, 349
genes in all.

In [ ]:
print('Signature matrix:', sig.shape[0], 'marker genes x', sig.shape[1], 'cell types')
sig.head()

Scaling each gene to a 0-to-1 range (so a gene with huge counts and a gene with
small counts are comparable) and sorting genes by the type they mark makes the
structure obvious: a bright block down the diagonal for each cell type, dark
everywhere else.

In [ ]:
scaled = sig.div(sig.max(axis=1).replace(0, 1), axis=0)
gene_order = np.argsort(sig.values.argmax(axis=1))
fig, ax = plt.subplots(figsize=(6, 7.5))
sns.heatmap(scaled.iloc[gene_order], cmap='magma', yticklabels=False,
            cbar_kws={'label': 'relative expression (0-1)'}, ax=ax)
ax.set(title='Reference signature (genes grouped by the cell type they mark)',
       xlabel='', ylabel='marker genes')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 4. Working backwards: deconvolution

Here is the intuition. A bulk measurement is roughly the average of the cell-type
profiles, weighted by how much of each type is present. If you know each profile
(the signature) and you can see the blend (the bulk), you can work backwards to the
weights, the way you might guess the ingredients of a smoothie from its taste.
Written as an equation:

$$\text{bulk expression} \;\approx\; \text{signature} \times \text{fractions}$$

We know the bulk and the signature, and we solve for the fractions. Two practical
details keep the answer sensible:

1. Fractions cannot be negative, so we use *non-negative least squares* (NNLS): it
   finds the fractions that best reproduce the bulk values without ever dropping
   below zero.
2. We first divide each gene by its largest value across cell types, putting every
   gene on the same 0-to-1 footing so a few loud genes do not drown out the rest.

The function below does both.

In [ ]:
def deconvolve(sig, bulk, scale_genes=True):
    # Estimate the fraction of each cell type in every bulk sample.
    shared = sorted(set(sig.index) & set(bulk.index))
    sig, bulk = sig.loc[shared], bulk.loc[shared]
    if scale_genes:
        scale = sig.max(axis=1).replace(0, 1)   # put each gene on a 0-1 scale
    else:
        scale = pd.Series(1.0, index=sig.index)
    S = sig.div(scale, axis=0).values
    out = {}
    for sample in bulk.columns:
        coef, _ = nnls(S, (bulk[sample] / scale).values)
        out[sample] = coef / coef.sum() if coef.sum() > 0 else coef
    return pd.DataFrame(out, index=sig.columns).T

print('deconvolve() defined. Shared genes with bulk:',
      len(set(sig.index) & set(bulk.index)))

### One tumor at a time

Start with a single tumor. The bars are its estimated cell-type fractions, which add
up to 1.

In [ ]:
patient = bulk.columns[0]
recipe = deconvolve(sig, bulk[[patient]]).iloc[0].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=recipe.values, y=recipe.index, color='#55a868', ax=ax)
ax.set(title=f'Estimated cell-type composition of tumor {patient}', xlabel='fraction')
sns.despine(); plt.tight_layout(); plt.show()
print('Fractions sum to', round(float(recipe.sum()), 3))

### Why the scaling matters

Skip that per-gene scaling and the loudest marker genes take over, dragging the
estimate toward the connective-tissue cells. Same tumor, solved both ways:

In [ ]:
with_scale = deconvolve(sig, bulk[[patient]], scale_genes=True).iloc[0]
no_scale   = deconvolve(sig, bulk[[patient]], scale_genes=False).iloc[0]
comp = pd.DataFrame({'with scaling': with_scale, 'without scaling': no_scale})
comp.plot.bar(figsize=(8, 4), color=['#55a868', '#c44e52'])
plt.ylabel('fraction'); plt.title(f'Effect of per-gene scaling (tumor {patient})')
plt.xticks(rotation=45, ha='right'); sns.despine(); plt.tight_layout(); plt.show()

**Exercise 2.** Set `patient_index` to any number from 0 to 1091 to look at a
different tumor.

In [ ]:
patient_index = 0       # <-- change to any number from 0 to 1091
p = bulk.columns[patient_index]
r = deconvolve(sig, bulk[[p]]).iloc[0].sort_values(ascending=False)
sns.barplot(x=r.values, y=r.index, color='#55a868')
plt.title(f'Tumor {p}'); plt.xlabel('fraction'); sns.despine(); plt.tight_layout(); plt.show()

## 5. The whole cohort at once

Run the same calculation across all ~1,000 tumors. Each row is a patient, each
column the estimated fraction of one cell type.

In [ ]:
fractions = deconvolve(sig, bulk)
print('Estimated fractions for', fractions.shape[0], 'tumors:')
fractions.round(3).head()

Plotting the spread of each fraction across the cohort shows how much tumors vary.
Cancer cells are usually the largest share, but the range is wide: some tumors are
mostly cancer, others mostly stroma and immune cells.

In [ ]:
long = fractions.melt(var_name='cell_type', value_name='fraction')
ordc = fractions.mean().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=long, x='fraction', y='cell_type', order=ordc, color='#8172b3', ax=ax)
ax.set(title='Distribution of estimated cell-type fractions across tumors', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()

The fractions are not independent. A tumor that is 80% cancer cells has only 20%
left for everything else, so the cell types compete for space. The heatmap below
shows which types rise and fall together across patients. The strongest signal is
the one you would expect: more cancer cells means less fibroblast and less
endothelium.

In [ ]:
corr = fractions.corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            square=True, cbar_kws={'label': 'correlation'}, ax=ax)
ax.set(title='Co-occurrence of cell types across tumors')
plt.tight_layout(); plt.show()

## 6. Lobular versus ductal

Now the comparison. Do lobular (ILC) and ductal (IDC) tumors differ in what they are
made of? We attach each patient's subtype to its fractions and compare the average
makeup.

In [ ]:
frac = fractions.copy()
frac['subtype'] = clin.set_index('sample')['subtype'].reindex(frac.index).values
frac = frac.dropna(subset=['subtype'])
mean_comp = frac.groupby('subtype')[list(fractions.columns)].mean()
mean_comp.plot.bar(stacked=True, colormap='tab10', figsize=(6, 5))
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
plt.ylabel('mean fraction'); plt.title('Average composition by subtype')
plt.xticks(rotation=0); sns.despine(); plt.tight_layout(); plt.show()

The averages look almost identical, so we need a real test. The **Mann-Whitney U
test** compares two groups without assuming a bell-curve distribution, and a small
p-value means the gap is unlikely to be chance. We are running seven tests at once,
one per cell type, which inflates the odds of a fluke, so we tighten the bar with a
**Bonferroni** correction: a result has to clear p < 0.05/7 (about 0.007) to count.

In [ ]:
from scipy.stats import mannwhitneyu
n_tests = len(fractions.columns)
threshold = 0.05 / n_tests
results = []
for ct in fractions.columns:
    a = frac[frac.subtype == 'ILC'][ct]
    b = frac[frac.subtype == 'IDC'][ct]
    U, p = mannwhitneyu(a, b)
    effect = 1 - 2 * U / (len(a) * len(b))     # rank-biserial effect size (-1..1)
    results.append({'cell_type': ct, 'IDC_mean': b.mean(), 'ILC_mean': a.mean(),
                    'p_value': p, 'effect_size': effect,
                    'significant': p < threshold})
results = pd.DataFrame(results).sort_values('p_value')
print(f'Bonferroni threshold: p < {threshold:.4f}')
results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=frac.melt(id_vars='subtype', value_vars=list(fractions.columns),
                           var_name='cell_type', value_name='fraction'),
            x='cell_type', y='fraction', hue='subtype', hue_order=['IDC', 'ILC'],
            palette=['#bbbbbb', '#d1495b'], ax=ax)
ax.set(title='Cell-type fractions by subtype', xlabel='')
plt.xticks(rotation=30, ha='right'); sns.despine(); plt.tight_layout(); plt.show()

**Exercise 3.** Pick any of the seven cell types and compare it between subtypes on
its own.

In [ ]:
cell_type = 'Endothelial'      # <-- change to any cell type from the table above
sns.boxplot(data=frac, x='subtype', y=cell_type, order=['IDC', 'ILC'],
            hue='subtype', legend=False, palette={'IDC': '#bbbbbb', 'ILC': '#d1495b'})
U, p = mannwhitneyu(frac[frac.subtype=='ILC'][cell_type], frac[frac.subtype=='IDC'][cell_type])
plt.title(f'{cell_type}  (p = {p:.1e})'); sns.despine(); plt.tight_layout(); plt.show()

## 7. The genes themselves, and a trap

Composition barely moves between the two subtypes, so that is not what makes lobular
cancer lobular. The difference is in the cancer cells. Here are six well-known
breast-cancer genes in the bulk data, on a log scale since the values span a wide
range.

In [ ]:
CANCER_GENES = ['CDH1', 'ESR1', 'ERBB2', 'MKI67', 'TP53', 'PIK3CA']
sub = clin.set_index('sample')['subtype']
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, g in zip(axes.ravel(), CANCER_GENES):
    if g not in bulk.index:
        ax.set_visible(False); continue
    df = pd.DataFrame({'expr': np.log1p(bulk.loc[g]),
                       'subtype': sub.reindex(bulk.columns).values}).dropna()
    p = mannwhitneyu(df[df.subtype=='ILC'].expr, df[df.subtype=='IDC'].expr).pvalue
    sns.boxplot(data=df, x='subtype', y='expr', order=['IDC', 'ILC'], hue='subtype',
                legend=False, palette={'IDC': '#bbbbbb', 'ILC': '#d1495b'}, ax=ax)
    ax.set(title=f'{g}  (p = {p:.0e})', xlabel='', ylabel='log expression')
sns.despine(); plt.tight_layout(); plt.show()

One gene jumps out: CDH1, much lower in lobular tumors. CDH1 codes for E-cadherin,
the protein that glues epithelial cells to their neighbors. Losing it is the
defining event in lobular carcinoma and the reason these tumors grow in thin
single-file strands instead of clumps.

Here is the trap that bulk data sets for you. A gene can look different between two
groups for two very different reasons:

1. the cells that make it genuinely turned it up or down (a change inside the
   cells), or
2. the tumors simply contain different amounts of the cells that make it (a change
   in the mix).

These mean completely different things biologically, and the bulk number alone
cannot separate them. Composition can. Start with a quick sanity check: if CDH1 were
low in lobular tumors only because they held fewer cancer cells, then the cancer-cell
fraction should be lower too.

In [ ]:
chk = frac.groupby('subtype').agg(cancer_fraction=('Cancer Epithelial', 'mean'))
chk['CDH1_expression'] = [bulk.loc['CDH1', frac[frac.subtype==s].index].mean()
                          for s in chk.index]
print(chk.round(3))

It is not. The cancer-cell fraction is essentially the same in both subtypes, yet
CDH1 runs several times lower in lobular. Same number of cancer cells, far less CDH1,
so the cells themselves are making less. That is a real, cell-intrinsic change.

We can see both stories side by side by plotting each gene against the fraction of
the cells that produce it, colored by subtype:
- CDH1 versus cancer-cell fraction: at any given fraction, lobular tumors sit lower.
  The subtype drives the difference, not the amount of cancer cells.
- PECAM1 versus endothelial fraction: both subtypes fall on the same rising line. The
  gene is just counting blood vessels.

In [ ]:
common = [s for s in frac.index if s in bulk.columns]
st = sub.reindex(common).values
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (gene, ctype, label) in zip(axes, [
        ('CDH1', 'Cancer Epithelial', 'CDH1 vs cancer-cell fraction (cell-intrinsic)'),
        ('PECAM1', 'Endothelial', 'PECAM1 vs blood-vessel fraction (composition)')]):
    for s, col in [('IDC', '#777777'), ('ILC', '#d1495b')]:
        m = st == s
        ax.scatter(frac.loc[common, ctype][m], bulk.loc[gene, common][m],
                   s=12, alpha=0.5, color=col, label=s)
    r = np.corrcoef(frac.loc[common, ctype], bulk.loc[gene, common])[0, 1]
    ax.set(xlabel=f'{ctype} fraction', ylabel=f'{gene} expression',
           title=f'{label}\nr = {r:.2f}')
    ax.legend(frameon=False)
sns.despine(); plt.tight_layout(); plt.show()

To put a number on it, we can ask how much of a gene's variation across tumors is
explained by the relevant cell-type fraction, as a squared correlation turned into a
percentage. A high number means the gene is mostly a stand-in for how much of that
cell type is around. A low number means something else is driving it, including
regulation inside the cells.

In [ ]:
def pct_variation_from_composition(gene, ctype):
    # squared correlation (r^2) between a gene and a cell-type fraction, as a percent
    g = bulk.loc[gene, common].values.astype(float)
    x = frac.loc[common, ctype].values.astype(float)
    return np.corrcoef(g, x)[0, 1] ** 2 * 100

for gene, ctype in [('CDH1', 'Cancer Epithelial'), ('PECAM1', 'Endothelial')]:
    pct = pct_variation_from_composition(gene, ctype)
    verdict = 'mostly composition' if pct > 30 else 'mostly cell-intrinsic'
    print(f'{gene:7}: {pct:4.0f}% of its variation is explained by {ctype} fraction  ->  {verdict}')

**Exercise 4.** Pick a gene and the cell type that makes it, and see how much of its
variation is just composition. Try `gene='COL1A1'` with `cell_type='Fibroblasts'`,
or `gene='ESR1'` with `cell_type='Cancer Epithelial'`. Which genes are mostly
bookkeeping for cell counts, and which are doing something on their own?

In [ ]:
gene = 'COL1A1'             # <-- change the gene
cell_type = 'Fibroblasts'   # <-- and the cell type that makes it
if gene in bulk.index:
    pct = pct_variation_from_composition(gene, cell_type)
    print(f'{gene}: {pct:.0f}% of its variation is explained by {cell_type} fraction')
else:
    print(gene, 'is not in the bulk dataset; try another gene.')

## 8. Recap and caveats

What you did: explored a labeled single-cell atlas and saw how marker genes define
cell types; built a reference signature and used non-negative least squares to
estimate the composition of about a thousand bulk tumors; compared lobular and ductal
tumors and found their cellular makeup is nearly the same; and showed that the drop
in CDH1 in lobular cancer is a real change inside the cells, while a gene like PECAM1
mostly just tracks how many blood vessels a tumor has.

The broader point is worth holding onto. A difference in bulk expression can be a
real change in the cells or just a different mix of cells, and you cannot tell which
from the bulk number alone. Estimating composition is one way to separate them, and
the question comes up any time bulk measurements are compared between groups.

A few caveats. Deconvolution gives estimates, not exact counts, and it is only as
good as the reference: a cell type missing from the reference cannot be estimated,
and rare types are estimated poorly. The reference and the bulk tumors come from
different patients, so systematic differences between datasets can bias the result.
And the bulk gene list here was trimmed for speed; a full analysis would use every
gene shared with the signature.

Worth trying: redo Section 6 with a Benjamini-Hochberg correction instead of
Bonferroni; run every cancer gene through the composition check from Exercise 4; or
relate the estimated immune fraction to CD274 (PD-L1), a gene tied to immunotherapy
response.